In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

# Configuración: archivos de entrada y salida.
# RELLENAR_CON_CERO = False deja los vacíos en blanco (BLANK en Power BI); True los pasa a 0
RUTA_ENTRADA = "/content/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR,_BÁSICA_Y_MEDIA_POR_MUNICIPIO_20261108.csv"
RUTA_CSV  = "/content/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR,_BÁSICA_Y_MEDIA_POR_MUNICIPIO_20261108.csv"
RUTA_XLSX = "/content/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR,_BÁSICA_Y_MEDIA_POR_MUNICIPIO_20261108.xlsx"
RELLENAR_CON_CERO = False

# Paso 0. Se lee todo como texto para no perder ceros a la izquierda ni formatos
datos = pd.read_csv(RUTA_ENTRADA, dtype=str)

# Paso 1. Se eliminan los espacios sobrantes en todas las columnas
for col in datos.columns:
    datos[col] = datos[col].str.strip()
print("Filas y columnas originales:", datos.shape)

# Paso 2. Se quitan las filas NACIONAL (totales del país, duplicarían las sumas)
datos = datos[datos["MUNICIPIO"] != "NACIONAL"].copy()

# Paso 3. Códigos con ceros a la izquierda (5 y 2 dígitos), como texto
datos["CÓDIGO_MUNICIPIO"]    = datos["CÓDIGO_MUNICIPIO"].str.zfill(5)
datos["CÓDIGO_DEPARTAMENTO"] = datos["CÓDIGO_DEPARTAMENTO"].str.zfill(2)
print("Control, municipios que no coinciden con su departamento (esperado 0):",
      (datos["CÓDIGO_MUNICIPIO"].str[:2] != datos["CÓDIGO_DEPARTAMENTO"]).sum())

# Paso 4. En 2021 el punto de miles se leyó como decimal: se corrige solo ese año
es_2021 = datos["AÑO"] == "2021"
fuera = (~es_2021) & (
    datos["POBLACIÓN_5_16"].str.contains(".", regex=False, na=False)
    | datos["CÓDIGO_ETC"].str.contains(".", regex=False, na=False)
)
print("Control, valores con punto fuera de 2021 (esperado 0):", fuera.sum())

datos["POBLACIÓN_5_16"] = datos["POBLACIÓN_5_16"].str.replace(",", "", regex=False)  # Bogotá: 1,174,274
pob = pd.to_numeric(datos["POBLACIÓN_5_16"], errors="coerce")
pob_punto = es_2021 & datos["POBLACIÓN_5_16"].str.contains(".", regex=False, na=False)
pob.loc[pob_punto] = pob.loc[pob_punto] * 1000
pob_chica = es_2021 & (pob < 100)   # tres municipios perdieron los ceros (13 era 13.000)
print("Poblaciones 2021 corregidas:", datos.loc[pob_chica, "MUNICIPIO"].tolist())
pob.loc[pob_chica] = pob.loc[pob_chica] * 1000
datos["POBLACIÓN_5_16"] = np.round(pob).astype("Int64")

etc = pd.to_numeric(datos["CÓDIGO_ETC"], errors="coerce")
etc_punto = es_2021 & datos["CÓDIGO_ETC"].str.contains(".", regex=False, na=False)
etc.loc[etc_punto] = etc.loc[etc_punto] * 1000
# CÓDIGO_ETC queda como TEXTO (es un identificador, no una cantidad)
datos["CÓDIGO_ETC"] = np.round(etc).astype("Int64").astype(str).replace("<NA>", pd.NA)

# Paso 5. Se corrigen los nombres de municipios por código (tildes o letras)
correcciones_municipios = {
    "05044":"Anzá","05237":"Donmatías","05264":"Entrerríos","05321":"Guatapé",
    "05665":"San Pedro de Urabá","05756":"Sonsón","13490":"Norosí","15296":"Gámeza",
    "15842":"Úmbita","18094":"Belén de los Andaquíes","19300":"Guachené","19517":"Páez",
    "19760":"Sotará","19821":"Toribío","25151":"Cáqueza","25279":"Fómeque",
    "25293":"Gachalá","25328":"Guayabal de Síquima","25426":"Machetá",
    "25843":"Villa de San Diego de Ubaté","27025":"Alto Baudó","27099":"Bojayá",
    "27150":"Carmen del Darién","27580":"Río Iró","41357":"Íquira","47258":"El Piñón",
    "47660":"Sabanas de San Ángel","52207":"Consacá","52427":"Magüí",
    "52835":"San Andrés de Tumaco","54003":"Ábrego",
    "54128":"Cáchira",        # estaba "Cachirá"
    "63130":"Calarcá","68298":"Gámbita",
    "68406":"Lebrija",        # estaba "Lebríja"
    "70823":"Tolú Viejo",     # estaba "Toluviejo"
    "76126":"Calima El Darién","85015":"Chámeza","97161":"Carurú",
}
nombre_nuevo = datos["CÓDIGO_MUNICIPIO"].map(correcciones_municipios)
datos["MUNICIPIO"] = nombre_nuevo.fillna(datos["MUNICIPIO"])

# Paso 6. Se unifican nombres repetidos de Bogotá, San Andrés, Quindío y algunas ETC
datos["MUNICIPIO"] = datos["MUNICIPIO"].replace({"Bogotá, D.C.": "Bogotá D.C."})
arch_mal = "Archipiélago de San Andrés. Providencia y Santa Catalina"
arch_ok  = "Archipiélago de San Andrés, Providencia y Santa Catalina"
datos["DEPARTAMENTO"] = datos["DEPARTAMENTO"].replace({
    "Bogotá, D.C.": "Bogotá D.C.", arch_mal: arch_ok, "Quindio": "Quindío"})
datos["ETC"] = datos["ETC"].replace({
    "Bogotá, D.C. (ETC)": "Bogotá D.C. (ETC)", arch_mal + " (ETC)": arch_ok + " (ETC)",
    "Quindio (ETC)": "Quindío (ETC)", "La estrella(ETC)": "La Estrella (ETC)",
    "Apartado": "Apartadó", "San Andres de Tumaco": "San Andrés de Tumaco"})

# Paso 7. Los indicadores pasan de texto ("56,11%") a número (56.11)
columnas_llave = ["AÑO","CÓDIGO_MUNICIPIO","MUNICIPIO","CÓDIGO_DEPARTAMENTO",
                  "DEPARTAMENTO","CÓDIGO_ETC","ETC","POBLACIÓN_5_16"]
columnas_indicadores = [c for c in datos.columns if c not in columnas_llave]

antes = datos[columnas_indicadores].notna().sum().sum()
for col in columnas_indicadores:
    datos[col] = datos[col].str.replace("%", "", regex=False).str.replace(",", ".", regex=False)
    datos[col] = np.round(pd.to_numeric(datos[col], errors="coerce"), 3)
print("Control, valores perdidos al convertir (esperado 0):",
      antes - datos[columnas_indicadores].notna().sum().sum())

# Paso 8. Filas con todos los indicadores en 0 o vacíos = "sin dato": se dejan vacías
sin_dato = (datos[columnas_indicadores].fillna(0) == 0).all(axis=1)
datos.loc[sin_dato, columnas_indicadores] = np.nan
print("Filas sin dato:", sin_dato.sum())

# Paso 9. El año pasa a entero y la tabla se ordena por año y municipio
datos["AÑO"] = datos["AÑO"].astype(int)
datos = datos.sort_values(["AÑO", "CÓDIGO_MUNICIPIO"]).reset_index(drop=True)

# Paso 10. Verificación final (cada control debe dar 0)
print("Filas:", datos.shape[0])
print("Control, duplicados año + municipio:", datos.duplicated(["AÑO", "CÓDIGO_MUNICIPIO"]).sum())
print("Control, municipios con más de un nombre:", (datos.groupby("CÓDIGO_MUNICIPIO")["MUNICIPIO"].nunique() > 1).sum())
print("Control, ETC con más de un nombre:", (datos.groupby("CÓDIGO_ETC")["ETC"].nunique() > 1).sum())

# Paso 11. Los vacíos quedan en blanco o en 0, según la configuración
if RELLENAR_CON_CERO:
    datos = datos.fillna(0)
print("Vacíos en el archivo:", datos.isna().sum().sum())

# Paso 12. Se guarda en CSV y en Excel (con filtros y encabezado fijo)
datos.to_csv(RUTA_CSV, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(RUTA_XLSX, engine="openpyxl") as writer:
    datos.to_excel(writer, sheet_name="Datos", index=False)
    hoja = writer.sheets["Datos"]
    hoja.freeze_panes = "D2"
    hoja.auto_filter.ref = hoja.dimensions

# Paso 13. Descarga en Colab (fuera de Colab los archivos quedan en la carpeta de trabajo)
try:
    from google.colab import files
    files.download(RUTA_XLSX)
    files.download(RUTA_CSV)
except ImportError:
    print("Archivos guardados:", RUTA_XLSX, "y", RUTA_CSV)

datos

Filas y columnas originales: (15707, 41)
Control, municipios que no coinciden con su departamento (esperado 0): 0
Control, valores con punto fuera de 2021 (esperado 0): 0
Poblaciones 2021 corregidas: ['San Onofre', 'Capitanejo', 'Filadelfia']
Control, valores perdidos al convertir (esperado 0): 0
Filas sin dato: 29
Filas: 15704
Control, duplicados año + municipio: 0
Control, municipios con más de un nombre: 0
Control, ETC con más de un nombre: 0
Vacíos en el archivo: 21124


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,...,REPROBACIÓN,REPROBACIÓN_TRANSICIÓN,REPROBACIÓN_PRIMARIA,REPROBACIÓN_SECUNDARIA,REPROBACIÓN_MEDIA,REPITENCIA,REPITENCIA_TRANSICIÓN,REPITENCIA_PRIMARIA,REPITENCIA_SECUNDARIA,REPITENCIA_MEDIA
0,2011,05001,Medellín,05,Antioquia,3759,Medellín,386466,108.73,108.50,...,0.03,0.00,0.00,0.01,0.17,4.57,0.15,3.26,7.44,2.95
1,2011,05002,Abejorral,05,Antioquia,3758,Antioquia (ETC),4146,97.81,97.80,...,1.70,0.00,1.23,2.96,1.18,0.89,0.00,0.85,1.08,1.23
2,2011,05004,Abriaquí,05,Antioquia,3758,Antioquia (ETC),483,88.61,88.60,...,7.29,0.00,1.47,14.66,7.46,1.69,3.13,1.47,2.22,0.00
3,2011,05021,Alejandría,05,Antioquia,3758,Antioquia (ETC),702,118.52,118.50,...,3.58,0.00,2.16,4.39,8.04,0.60,0.00,0.96,0.00,0.00
4,2011,05030,Amagá,05,Antioquia,3758,Antioquia (ETC),6631,78.65,78.70,...,8.99,0.24,6.73,14.46,7.45,0.42,0.00,0.24,0.91,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15699,2024,97889,Yavaraté,97,Vaupés,3831,Vaupés (ETC),562,46.26,46.26,...,8.74,0.00,15.29,1.33,0.00,7.69,0.00,8.92,8.00,8.33
15700,2024,99001,Puerto Carreño,99,Vichada,3832,Vichada (ETC),4600,148.35,148.17,...,10.08,0.00,10.86,12.92,1.83,12.72,1.90,13.86,14.92,4.36
15701,2024,99524,La Primavera,99,Vichada,3832,Vichada (ETC),2645,90.96,90.96,...,8.83,0.00,11.00,8.84,2.91,13.15,0.68,17.38,11.67,3.64
15702,2024,99624,Santa Rosalía,99,Vichada,3832,Vichada (ETC),1157,81.42,81.42,...,7.63,0.00,7.12,10.92,5.43,13.17,0.00,15.00,16.67,2.17
